# 06 — Dashboard Validation

Notebook này dùng để **đối chiếu các KPI trong Power BI với dữ liệu đã export**.

## Mục tiêu

- Kiểm tra Sales KPI: Net Sales, Cost, Profit, Profit Margin, Quantity Sold.
- Kiểm tra Return KPI: Return Quantity, Gross Sold Quantity, Return Rate.
- Kiểm tra Inventory KPI tại snapshot mới nhất.
- Kiểm tra filter theo Brand (ví dụ `Brand1`).
- Kiểm tra tuần bất thường `202153`.
- Tạo bảng `PASS / FAIL` để dùng trong README/GitHub.

> Trước khi nhập số từ Power BI, hãy clear mọi selection trên chart.  
> Brand = All, Channel = All, Year = 2022 + 2023.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Cấu trúc project mong đợi:
# project/
# ├── notebooks/06_dashboard_validation.ipynb
# └── data/processed/powerbi_data/*.csv

POWERBI_PATH = Path("../data/processed/powerbi_data")

# Fallback nếu notebook được chạy từ project root
if not POWERBI_PATH.exists():
    POWERBI_PATH = Path("data/processed/powerbi_data")

print("POWERBI_PATH:", POWERBI_PATH.resolve())
print("Exists:", POWERBI_PATH.exists())


POWERBI_PATH: /Users/mac/sales-inventory-dashboard/data/processed/powerbi_data
Exists: True


In [2]:
required_files = [
    "DimProduct.csv",
    "DimCustomer.csv",
    "DimSite.csv",
    "DimWeek.csv",
    "DimDate.csv",
    "DimPlant.csv",
    "FactSales.csv",
    "FactInventory.csv",
]

missing = [f for f in required_files if not (POWERBI_PATH / f).exists()]

if missing:
    raise FileNotFoundError(
        "Thiếu các file Power BI export sau:\n- "
        + "\n- ".join(missing)
        + "\n\nHãy chạy notebook 05_powerbi_data_export trước."
    )

print("Đủ 8 file Power BI export.")


Đủ 8 file Power BI export.


In [3]:
fact_sales = pd.read_csv(POWERBI_PATH / "FactSales.csv", low_memory=False)
fact_inventory = pd.read_csv(POWERBI_PATH / "FactInventory.csv", low_memory=False)
dim_product = pd.read_csv(POWERBI_PATH / "DimProduct.csv", low_memory=False)
dim_date = pd.read_csv(POWERBI_PATH / "DimDate.csv", low_memory=False)
dim_week = pd.read_csv(POWERBI_PATH / "DimWeek.csv", low_memory=False)

print("FactSales:", fact_sales.shape)
print("FactInventory:", fact_inventory.shape)
print("DimProduct:", dim_product.shape)
print("DimDate:", dim_date.shape)
print("DimWeek:", dim_week.shape)


FactSales: (831966, 27)
FactInventory: (1367080, 13)
DimProduct: (94868, 26)
DimDate: (335, 11)
DimWeek: (86, 7)


## 1. Validate Sales KPI — Full dataset

Các công thức dưới đây phải tương ứng với DAX:

```DAX
Total Net Sales = SUM(FactSales[net_price])
Total Cost = SUM(FactSales[cost_price])
Total Profit = SUM(FactSales[profit])
Profit Margin = DIVIDE([Total Profit], [Total Net Sales], 0)
Total Quantity Sold = SUM(FactSales[sold_quantity])
```


In [4]:
sales_kpi = {
    "Total Net Sales": fact_sales["net_price"].sum(),
    "Total Cost": fact_sales["cost_price"].sum(),
    "Total Profit": fact_sales["profit"].sum(),
    "Total Quantity Sold": fact_sales["sold_quantity"].sum(),
}

sales_kpi["Profit Margin"] = (
    sales_kpi["Total Profit"] / sales_kpi["Total Net Sales"]
    if sales_kpi["Total Net Sales"] != 0 else 0
)

pd.Series(sales_kpi)


Total Net Sales       332,230,025,498.0000
Total Cost            254,170,428,358.0000
Total Profit           78,059,597,140.0000
Total Quantity Sold         1,174,515.0000
Profit Margin                       0.2350
dtype: float64

### Expected full-context values đã quan sát trong Power BI

- Total Net Sales ≈ **332.23bn**
- Total Cost ≈ **254.17bn**
- Total Profit ≈ **78.06bn**
- Profit Margin ≈ **23.5%**
- Total Quantity Sold ≈ **1.17M**

Ta dùng số nguyên gốc để validation, không so sánh chuỗi đã format.


In [5]:
powerbi_expected_sales = {
    "Total Net Sales": 332_230_025_498,
    "Total Cost": 254_170_428_358,
    "Total Profit": 332_230_025_498 - 254_170_428_358,
    "Profit Margin": (332_230_025_498 - 254_170_428_358) / 332_230_025_498,
    "Total Quantity Sold": 1_174_515,
}

rows = []
for metric, expected in powerbi_expected_sales.items():
    actual = sales_kpi[metric]
    # tiền/số lượng: tolerance tuyệt đối nhỏ; tỷ lệ: tolerance 1e-12
    tol = 1e-12 if metric == "Profit Margin" else 0.5
    diff = actual - expected
    rows.append({
        "Metric": metric,
        "Python": actual,
        "Power BI Expected": expected,
        "Difference": diff,
        "Status": "PASS" if abs(diff) <= tol else "FAIL",
    })

sales_validation = pd.DataFrame(rows)
sales_validation


,Metric,Python,Power BI Expected,Difference,Status
0,Total Net Sales,"332,230,025,498.0000","332,230,025,498.0000",0.0000,PASS
1,Total Cost,"254,170,428,358.0000","254,170,428,358.0000",0.0000,PASS
2,Total Profit,"78,059,597,140.0000","78,059,597,140.0000",0.0000,PASS
3,Profit Margin,0.2350,0.2350,0.0000,PASS
4,Total Quantity Sold,"1,174,515.0000","1,174,515.0000",0.0000,PASS


## 2. Validate Return KPI

Negative `sold_quantity` được giữ lại vì đây là return/reversal.

Công thức validation:

- **Return Quantity** = trị tuyệt đối tổng quantity âm.
- **Gross Sold Quantity** = tổng quantity dương.
- **Return Rate** = Return Quantity / Gross Sold Quantity.

Nếu DAX trong Power BI của bạn đang dùng đúng định nghĩa này thì các kết quả phải khớp.


In [6]:
return_quantity = abs(
    fact_sales.loc[fact_sales["sold_quantity"] < 0, "sold_quantity"].sum()
)

gross_sold_quantity = fact_sales.loc[
    fact_sales["sold_quantity"] > 0, "sold_quantity"
].sum()

return_rate = (
    return_quantity / gross_sold_quantity
    if gross_sold_quantity != 0 else 0
)

return_validation = pd.DataFrame({
    "Metric": ["Return Quantity", "Gross Sold Quantity", "Return Rate"],
    "Python": [return_quantity, gross_sold_quantity, return_rate]
})

return_validation


,Metric,Python
0,Return Quantity,"27,643.0000"
1,Gross Sold Quantity,"1,202,158.0000"
2,Return Rate,0.0230


In [7]:
print(f"Return Quantity     : {return_quantity:,}")
print(f"Gross Sold Quantity : {gross_sold_quantity:,}")
print(f"Return Rate         : {return_rate:.4%}")


Return Quantity     : 27,643
Gross Sold Quantity : 1,202,158
Return Rate         : 2.2994%


## 3. Validate Inventory KPI tại latest snapshot

Ta join `FactInventory[date_key]` với `DimDate[date_key]`.

Các KPI:

- Latest Inventory Date
- Current Inventory Quantity
- Products in Latest Snapshot
- Products in Stock
- Out of Stock Products

> `Products in Stock` được tính ở **product grain**: aggregate quantity theo product trước, rồi mới kiểm tra `> 0`.


In [8]:
# Tìm cột ngày thực tế trong DimDate
date_candidates = [
    "date", "full_date", "calendar_date", "snapshot_date"
]

date_col = next(
    (c for c in date_candidates if c in dim_date.columns),
    None
)

if date_col is None:
    # Nếu không đúng tên phổ biến, tìm cột parse được nhiều giá trị date nhất
    best_col = None
    best_count = -1
    for c in dim_date.columns:
        parsed = pd.to_datetime(dim_date[c], errors="coerce")
        cnt = parsed.notna().sum()
        if cnt > best_count:
            best_col = c
            best_count = cnt
    date_col = best_col

dim_date["_validation_date"] = pd.to_datetime(
    dim_date[date_col], errors="coerce"
)

inv = fact_inventory.merge(
    dim_date[["date_key", "_validation_date"]],
    on="date_key",
    how="left",
    validate="many_to_one"
)

latest_inventory_date = inv["_validation_date"].max()
latest_inv = inv[inv["_validation_date"] == latest_inventory_date].copy()

inventory_quantity = latest_inv["quantity"].sum()

product_qty = (
    latest_inv.groupby("product_key", dropna=False)["quantity"]
    .sum()
)

products_in_latest_snapshot = product_qty.index.nunique()
products_in_stock = (product_qty > 0).sum()
out_of_stock_products = (product_qty <= 0).sum()

inventory_kpi = pd.Series({
    "Latest Inventory Date": latest_inventory_date,
    "Current Inventory Quantity": inventory_quantity,
    "Products in Latest Snapshot": products_in_latest_snapshot,
    "Products in Stock": products_in_stock,
    "Out of Stock Products": out_of_stock_products,
})

inventory_kpi


Latest Inventory Date          2022-12-31 00:00:00
Current Inventory Quantity                   93317
Products in Latest Snapshot                  12975
Products in Stock                            12975
Out of Stock Products                            0
dtype: object

### Expected full-context Inventory values

Kết quả Power BI full context đã được kiểm tra trước đó:

- Latest Inventory Date = **2022-12-31**
- Current Inventory Quantity = **93,317**
- Products in Latest Snapshot = **12,975**
- Products in Stock = **12,975**
- Out of Stock Products = **0**


In [9]:
powerbi_expected_inventory = {
    "Latest Inventory Date": pd.Timestamp("2022-12-31"),
    "Current Inventory Quantity": 93_317,
    "Products in Latest Snapshot": 12_975,
    "Products in Stock": 12_975,
    "Out of Stock Products": 0,
}

inv_rows = []

for metric, expected in powerbi_expected_inventory.items():
    actual = inventory_kpi[metric]

    if metric == "Latest Inventory Date":
        diff = pd.Timestamp(actual) - expected
        status = "PASS" if pd.Timestamp(actual) == expected else "FAIL"
    else:
        diff = actual - expected
        status = "PASS" if diff == 0 else "FAIL"

    inv_rows.append({
        "Metric": metric,
        "Python": actual,
        "Power BI Expected": expected,
        "Difference": diff,
        "Status": status,
    })

inventory_validation = pd.DataFrame(inv_rows)
inventory_validation


,Metric,Python,Power BI Expected,Difference,Status
0,Latest Inventory Date,2022-12-31 00:00:00,2022-12-31 00:00:00,0 days 00:00:00,PASS
1,Current Inventory Quantity,93317,93317,0,PASS
2,Products in Latest Snapshot,12975,12975,0,PASS
3,Products in Stock,12975,12975,0,PASS
4,Out of Stock Products,0,0,0,PASS


## 4. Validate Brand filter

Đây là test quan trọng cho relationship:

`DimProduct[product_key] 1:* FactSales[product_key]`  
`DimProduct[product_key] 1:* FactInventory[product_key]`

Ta kiểm tra `Brand1` để chắc chắn Brand filter tác động cả Sales và Inventory.


In [10]:
brand_col = "brand_name"

brand_name = "Brand1"

brand_keys = set(
    dim_product.loc[
        dim_product[brand_col].astype(str) == brand_name,
        "product_key"
    ]
)

sales_brand = fact_sales[
    fact_sales["product_key"].isin(brand_keys)
].copy()

inv_brand = latest_inv[
    latest_inv["product_key"].isin(brand_keys)
].copy()

brand_sales_net = sales_brand["net_price"].sum()
brand_sales_cost = sales_brand["cost_price"].sum()
brand_sales_profit = sales_brand["profit"].sum()
brand_sales_margin = (
    brand_sales_profit / brand_sales_net if brand_sales_net != 0 else 0
)
brand_net_quantity = sales_brand["sold_quantity"].sum()

brand_return_qty = abs(
    sales_brand.loc[
        sales_brand["sold_quantity"] < 0, "sold_quantity"
    ].sum()
)
brand_gross_qty = sales_brand.loc[
    sales_brand["sold_quantity"] > 0, "sold_quantity"
].sum()
brand_return_rate = (
    brand_return_qty / brand_gross_qty if brand_gross_qty != 0 else 0
)

brand_product_qty = inv_brand.groupby("product_key")["quantity"].sum()

brand_inventory_quantity = inv_brand["quantity"].sum()
brand_products_snapshot = brand_product_qty.index.nunique()
brand_products_stock = (brand_product_qty > 0).sum()
brand_oos = (brand_product_qty <= 0).sum()

brand1_validation = pd.Series({
    "Total Net Sales": brand_sales_net,
    "Total Cost": brand_sales_cost,
    "Total Profit": brand_sales_profit,
    "Profit Margin": brand_sales_margin,
    "Total Quantity Sold": brand_net_quantity,
    "Return Quantity": brand_return_qty,
    "Return Rate": brand_return_rate,
    "Current Inventory Quantity": brand_inventory_quantity,
    "Products in Latest Snapshot": brand_products_snapshot,
    "Products in Stock": brand_products_stock,
    "Out of Stock Products": brand_oos,
})

brand1_validation


Total Net Sales               218,118,068,606.0000
Total Cost                    169,258,523,121.0000
Total Profit                   48,859,545,485.0000
Profit Margin                               0.2240
Total Quantity Sold                 1,004,744.0000
Return Quantity                        21,777.0000
Return Rate                                 0.0212
Current Inventory Quantity             72,110.0000
Products in Latest Snapshot            10,601.0000
Products in Stock                      10,601.0000
Out of Stock Products                       0.0000
dtype: float64

Trong Power BI, khi **Brand1 thực sự được filter/selected**, dashboard đã từng hiển thị xấp xỉ:

- Net Sales: 218.12bn
- Cost: 169.26bn
- Profit: 48.86bn
- Margin: 22.4%
- Quantity: 1.00M
- Return Quantity: 22K
- Return Rate: khoảng 2.1%
- Inventory Quantity: 72.11K
- Products in Stock / Snapshot: 10.60K
- OOS: 0

Do card đang format theo bn/M/K, Python sẽ cho số chính xác hơn.


## 5. Audit tuần bất thường `202153`

`202153` được giữ trong FactSales để không làm mất dữ liệu, nhưng ISO week này không có `week_start_date` hợp lệ trong DimWeek.

Vì vậy:

- dữ liệu vẫn nằm trong total KPI;
- Weekly Trend dùng `DimWeek[week_start_date]` sẽ không plot các dòng này.


In [11]:
invalid_week_sales = fact_sales[
    fact_sales["year_week"].astype(str) == "202153"
].copy()

invalid_week_audit = pd.Series({
    "Rows": len(invalid_week_sales),
    "Net Sales": invalid_week_sales["net_price"].sum(),
    "Cost": invalid_week_sales["cost_price"].sum(),
    "Quantity": invalid_week_sales["sold_quantity"].sum(),
})

invalid_week_audit


Rows               4647
Net Sales    1572419878
Cost         1123802680
Quantity           5778
dtype: int64

## 6. Final PASS / FAIL summary

Nếu tất cả mục chính là `PASS`, dashboard có thể được xem là đã validate ở mức KPI.


In [12]:
final_checks = pd.concat([
    sales_validation[["Metric", "Status"]].assign(Section="Sales"),
    inventory_validation[["Metric", "Status"]].assign(Section="Inventory"),
], ignore_index=True)

final_checks = final_checks[["Section", "Metric", "Status"]]

print(final_checks.to_string(index=False))

if (final_checks["Status"] == "PASS").all():
    print("\n✅ ALL CORE KPI CHECKS PASSED")
else:
    print("\n❌ CÓ KPI CHƯA KHỚP — cần kiểm tra filter context hoặc DAX.")


  Section                      Metric Status
    Sales             Total Net Sales   PASS
    Sales                  Total Cost   PASS
    Sales                Total Profit   PASS
    Sales               Profit Margin   PASS
    Sales         Total Quantity Sold   PASS
Inventory       Latest Inventory Date   PASS
Inventory  Current Inventory Quantity   PASS
Inventory Products in Latest Snapshot   PASS
Inventory           Products in Stock   PASS
Inventory       Out of Stock Products   PASS

✅ ALL CORE KPI CHECKS PASSED


In [13]:
# Lưu bảng validation để dùng trong README/GitHub
OUTPUT_PATH = Path("../data/processed/validation")
if not OUTPUT_PATH.parent.exists():
    OUTPUT_PATH = Path("data/processed/validation")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

sales_validation.to_csv(
    OUTPUT_PATH / "sales_kpi_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

inventory_validation.to_csv(
    OUTPUT_PATH / "inventory_kpi_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

invalid_week_audit.rename("Value").to_csv(
    OUTPUT_PATH / "invalid_week_202153_audit.csv",
    encoding="utf-8-sig"
)

print("Saved validation files to:", OUTPUT_PATH.resolve())


Saved validation files to: /Users/mac/sales-inventory-dashboard/data/processed/validation


## Kết luận Phase 2

Khi notebook chạy xong:

1. `Sales` core KPI phải PASS.
2. `Inventory` core KPI phải PASS.
3. Brand1 test phải gần đúng với card Power BI sau format.
4. `202153` phải được audit riêng, không xóa khỏi FactSales.
5. Không dùng `Current Inventory Value` trong dashboard vì semantics của `total_amount` chưa được xác nhận.

Sau đó project chuyển sang **Phase 3 — GitHub + README + documentation**.
